In [ ]:
!pip install -r req.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.0/388.0 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.4/285.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

train_rlvr_code

In [ ]:
import os
import sys
import torch
import re
import subprocess
import textwrap
import multiprocessing

# --- 1. VACINAS DE ESTABILIDADE (RTX 3060 + WINDOWS) ---
torch.cuda.is_bf16_supported = lambda: False
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"

from transformers import modeling_utils
_original_to = modeling_utils.PreTrainedModel.to
def _patched_to(self, *args, **kwargs):
    try:
        return _original_to(self, *args, **kwargs)
    except ValueError as e:
        raise e
modeling_utils.PreTrainedModel.to = _patched_to
# ---------------------------------------------------------------

from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- CONFIGURAÇÕES ---
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUTPUT_DIR = "qwen-rlvr-v2"

# --- 2. EXECUÇÃO SEGURA VIA SUBPROCESS ---
# Motivo da troca de multiprocessing para subprocess:
# No Windows (spawn), multiprocessing.Process re-importa o módulo __main__ inteiro
# para encontrar o _worker, executando patches de CUDA e imports pesados no processo
# filho — que crasha silenciosamente antes de colocar qualquer coisa na fila.
# subprocess.run spawna um Python limpo que só executa o código, sem nenhum import
# do módulo principal.

def run_code_safe(code_string, timeout=5.0):
    """
    Executa code_string em um processo Python completamente limpo via subprocess.
    Retorna True se o código rodar sem exceções, False se falhar ou timeout.
    """
    try:
        result = subprocess.run(
            [sys.executable, "-c", code_string],
            timeout=timeout,
            capture_output=True,
            text=True,
            stdin=subprocess.DEVNULL,   # [FIX BUG 1] input() recebe EOF imediato
        )                               # em vez de bloquear até o timeout
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    except Exception:
        return False

# --- 3. FUNÇÃO DE RECOMPENSA EXTERNA (RLVR) ---
def reward_rlvr_execution(completions, test_list, **kwargs):
    """
    A essência do RLVR: a recompensa vem do ambiente (testes unitários), não da probabilidade.
    """
    rewards = []

    signatures = kwargs.get("signature", [None] * len(completions))

    for idx, (completion, tests, sig) in enumerate(zip(completions, test_list, signatures)):
        # 1. Limpa a saída do modelo (remove Markdown)
        code = completion.strip()
        if "```python" in code:
            code = code.split("```python")[1].split("```")[0]
        elif "```" in code:
            code = code.split("```")[1].split("```")[0]

        code = textwrap.dedent(code).strip()

        # [FIX BUG 2] Fallback: se o modelo ainda gerar só o corpo sem def,
        # reconstrói a função completa usando a assinatura do dataset.
        if code and not re.match(r"^\s*(def |class |import |from )", code):
            if sig:
                indented_body = "\n".join("    " + line for line in code.splitlines())
                code = f"{sig}\n{indented_body}"

        # 2. Prepara o código com os testes do MBPP (ex: assert add(1,2) == 3)
        tests_str = "\n".join(tests)
        full_execution_code = f"{code}\n\n{tests_str}"

        # 3. Executa e recompensa
        passou = run_code_safe(full_execution_code)

        if passou:
            rewards.append(1.0)   # Recompensa positiva (feedback externo de sucesso)
        else:
            rewards.append(-1.0)  # Punição (erro de lógica, compilação ou loop infinito)

        # Monitor — mostra status do primeiro exemplo do batch
        if idx == 0:
            status = "✅ PASS" if passou else "❌ FAIL"
            print(f"\nMONITOR RLVR | {status}")
            print("-" * 50)
            print(code.strip()[:400])
            print("-" * 50 + "\n")

    return rewards

# --- 4. PREPARAÇÃO DOS DADOS ---
def format_data_func(example):
    gold_code = example.get('code', '')
    match = re.search(r"def\s+\w+\s*\(.*?\):", gold_code, re.DOTALL)
    signature = match.group(0) if match else "def solution():"

    prompt_raw = example.get('prompt', example.get('text', ''))

    # [FIX BUG 1] Proibir input() explicitamente no system prompt.
    # Sem isso, o modelo gerava funções com input() que bloqueavam o subprocess
    # esperando stdin que nunca chega, desperdiçando o timeout inteiro.
    # [FIX BUG 2] Pedir a função COMPLETA (def + corpo) em vez de só o corpo.
    # O prompt anterior ("ONLY with the indented function body") fazia o modelo
    # gerar apenas o corpo indentado sem o def — causando SyntaxError no exec.
    system_msg = (
        "You are a Python coding engine. "
        "Respond with the COMPLETE Python function, including the def line and the indented body. "
        "Do NOT use input() or any form of user input. "
        "All values must come from function parameters. "
        "Do NOT include any explanation or markdown, only the raw Python function."
    )

    prompt_text = (
        f"<|im_start|>system\n{system_msg}<|im_end|>\n"
        f"<|im_start|>user\nImplement:\n{signature}\n\"\"\"\n{prompt_raw}\n\"\"\"\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    return {
        "prompt": prompt_text,
        "test_list": example['test_list'],
        "signature": signature,   # guardamos a assinatura para o fallback abaixo
    }

# --- 5. EXECUÇÃO PRINCIPAL ---
def main():
    print(f"--- INICIANDO TREINO {OUTPUT_DIR} (RLVR v2 | subprocess | Verificador Externo) ---")

    # Seed global para reprodutibilidade entre runs
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    peft_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
        task_type=TaskType.CAUSAL_LM,
        lora_dropout=0.05,
    )

    training_args = GRPOConfig(
        output_dir=OUTPUT_DIR,
        learning_rate=5e-5,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_generations=4,
        max_prompt_length=600,
        max_completion_length=512,
        logging_steps=5,
        save_steps=50,
        fp16=True,
        beta=0.0005,
        use_vllm=False,
        report_to="none",
        temperature=0.7,
        seed=42,
        gradient_checkpointing=True,
        dataloader_pin_memory=False,
    )

    print("Carregando Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Carregando Modelo (FP16)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.pad_token_id = tokenizer.pad_token_id

    print("Carregando Dataset (MBPP)...")
    dataset = load_dataset("mbpp", "sanitized", split="train")
    dataset = dataset.map(format_data_func, batched=False)

    print("Iniciando Trainer...")
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_rlvr_execution,
        args=training_args,
        train_dataset=dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    print("TREINO RLVR INICIADO!")
    trainer.train()

    print("Salvando modelo final...")
    trainer.save_model(OUTPUT_DIR)
    print(f"Modelo RLVR salvo com sucesso em: {OUTPUT_DIR}")

if __name__ == "__main__":
    multiprocessing.freeze_support()
    main()

--- INICIANDO TREINO qwen-rlvr-v2 (RLVR v2 | subprocess | Verificador Externo) ---
Carregando Tokenizer...


<string>:196: FutureWarning: The `max_prompt_length` argument is deprecated and will be removed in version 0.28.0. You should instead filter your dataset before training to ensure that prompts do not exceed your desired length.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


TREINO RLVR INICIADO!

MONITOR RLVR | ✅ PASS
--------------------------------------------------
def remove_nested(test_tup):
    """
    Write a function to remove tuples from the given tuple.
    """
    result = tuple(item for item in test_tup if not isinstance(item, tuple))
    return result
--------------------------------------------------



Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
5,0.014700
10,0.007300
15,-0.033000
20,0.139600
25,-0.013700
30,-0.025500
35,0.010600
40,0.005000
45,0.014800
50,0.007700



MONITOR RLVR | ❌ FAIL
--------------------------------------------------
def cummulative_sum(test_list):
    """
    Write a function to find the cumulative sum of all the values that are present in the given tuple list.
    """
    cumulative_sum = 0
    for value in test_list:
        cumulative_sum += value
    return cumulative_sum
--------------------------------------------------


MONITOR RLVR | ✅ PASS
--------------------------------------------------
def noprofit_noloss(actual_cost, sale_amount):
    """
    Write a function to check whether the given amount has no profit and no loss
    """
    if actual_cost == sale_amount:
        return True
    else:
        return False
--------------------------------------------------


MONITOR RLVR | ✅ PASS
--------------------------------------------------
def count_same_pair(nums1, nums2):
    """
    The input is defined as two lists of the same length. Write a function to count indices where the lists have the same values.
    ""

avaliar rlvr_code humaneval

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from human_eval.data import write_jsonl, read_problems
from tqdm import tqdm
import re

# --- CONFIGURAÇÕES ---
BASE_MODEL   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_PATH = "qwen-rlvr-v2"   # pasta salva pelo treino RLVR
OUTPUT_FILE  = "samples_rlvr.jsonl"
DEVICE = "cuda"

# --- SYSTEM PROMPT UNIFICADO ---
# Idêntico nos três scripts para garantir comparação justa.
SYSTEM_MSG = (
    "You are a Python coding engine. "
    "Respond with the COMPLETE Python function, including the def line and the indented body. "
    "Do NOT use input() or any form of user input. "
    "All values must come from function parameters. "
    "Do NOT include any explanation or markdown, only the raw Python function."
)

# --- FUNÇÃO DE PARSING UNIFICADA ---
# Idêntica nos três scripts.
def tratar_codigo(prompt_original, completion):
    # 1. Remove Markdown
    if "```python" in completion:
        completion = completion.split("```python")[1].split("```")[0]
    elif "```" in completion:
        completion = completion.split("```")[1].split("```")[0]

    completion = completion.strip()

    # 2. Descobre o nome da função pelo prompt do HumanEval
    match_def = re.search(r"def\s+(\w+)\s*\(", prompt_original)
    nome_funcao = match_def.group(1) if match_def else None

    linhas = completion.split('\n')

    # Caso A: modelo gerou a assinatura completa ("def func():")
    # Extrai só o corpo e reindenta com 4 espaços.
    if nome_funcao and any(nome_funcao in l and "def " in l for l in linhas):
        corpo = []
        capturando = False
        indentacao_base = None

        for linha in linhas:
            if f"def {nome_funcao}" in linha:
                capturando = True
                continue

            if capturando:
                if indentacao_base is None and linha.strip():
                    indentacao_base = len(linha) - len(linha.lstrip())

                if indentacao_base is not None:
                    if len(linha) - len(linha.lstrip()) >= indentacao_base or linha.strip() == "":
                        corpo.append(linha[indentacao_base:])
                    else:
                        break

        completion_limpa = "\n".join(["    " + l for l in corpo])

    # Caso B: modelo gerou só o corpo sem assinatura
    else:
        if linhas and not linhas[0].startswith("    "):
            completion_limpa = "\n".join(["    " + l for l in linhas])
        else:
            completion_limpa = completion

    return completion_limpa

# --- CARREGAMENTO ---
print(f"Carregando tokenizer: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print(f"Carregando modelo base: {BASE_MODEL}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
# resize_token_embeddings removido — nenhum token novo foi adicionado no treino

print(f"Acoplando adaptador RLVR: {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

problems = read_problems()
samples = []

print("Gerando amostras (greedy, temp=0.0)...")
for task_id in tqdm(problems):
    problem = problems[task_id]

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"Implement:\n{problem['prompt']}"},
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_input, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.0,   # greedy — padrão da literatura para pass@1
            do_sample=False,
        )

    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in raw_output:
        raw_completion = raw_output.split("assistant")[-1]
    else:
        raw_completion = raw_output

    final_completion = tratar_codigo(problem['prompt'], raw_completion)
    samples.append(dict(task_id=task_id, completion=final_completion))

write_jsonl(OUTPUT_FILE, samples)
print(f"Arquivo {OUTPUT_FILE} gerado com {len(samples)} amostras.")

Carregando tokenizer: Qwen/Qwen2.5-Coder-1.5B-Instruct...
Carregando modelo base: Qwen/Qwen2.5-Coder-1.5B-Instruct...
Acoplando adaptador RLVR: qwen-rlvr-v2...
Gerando amostras (greedy, temp=0.0)...


100%|██████████| 164/164 [47:06<00:00, 17.23s/it]

Arquivo samples_rlvr.jsonl gerado com 164 amostras.


In [ ]:
!evaluate_functional_correctness samples_rlvr.jsonl

Reading samples...
164it [00:00, 7221.92it/s]
Running test suites...
100% 164/164 [00:01<00:00, 131.51it/s]
Writing results to samples_rlvr.jsonl_results.jsonl...
100% 164/164 [00:00<00:00, 54648.91it/s]
{'pass@1': np.float64(0.6707317073170732)}


resultados rlvr_code humaneval;

1.   teste 1: 0.7073170731707317;
2.   teste 2: 0.6707317073170732;

avaliando base humaneval

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from human_eval.data import write_jsonl, read_problems
from tqdm import tqdm
import re

# --- CONFIGURAÇÕES ---
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUTPUT_FILE = "samples_base.jsonl"
DEVICE = "cuda"

# --- SYSTEM PROMPT UNIFICADO ---
# Idêntico nos três scripts para garantir comparação justa.
SYSTEM_MSG = (
    "You are a Python coding engine. "
    "Respond with the COMPLETE Python function, including the def line and the indented body. "
    "Do NOT use input() or any form of user input. "
    "All values must come from function parameters. "
    "Do NOT include any explanation or markdown, only the raw Python function."
)

# --- FUNÇÃO DE PARSING UNIFICADA ---
# Idêntica nos três scripts.
def tratar_codigo(prompt_original, completion):
    # 1. Remove Markdown
    if "```python" in completion:
        completion = completion.split("```python")[1].split("```")[0]
    elif "```" in completion:
        completion = completion.split("```")[1].split("```")[0]

    completion = completion.strip()

    # 2. Descobre o nome da função pelo prompt do HumanEval
    match_def = re.search(r"def\s+(\w+)\s*\(", prompt_original)
    nome_funcao = match_def.group(1) if match_def else None

    linhas = completion.split('\n')

    # Caso A: modelo gerou a assinatura completa ("def func():")
    # Extrai só o corpo e reindenta com 4 espaços.
    if nome_funcao and any(nome_funcao in l and "def " in l for l in linhas):
        corpo = []
        capturando = False
        indentacao_base = None

        for linha in linhas:
            if f"def {nome_funcao}" in linha:
                capturando = True
                continue

            if capturando:
                if indentacao_base is None and linha.strip():
                    indentacao_base = len(linha) - len(linha.lstrip())

                if indentacao_base is not None:
                    if len(linha) - len(linha.lstrip()) >= indentacao_base or linha.strip() == "":
                        corpo.append(linha[indentacao_base:])
                    else:
                        break

        completion_limpa = "\n".join(["    " + l for l in corpo])

    # Caso B: modelo gerou só o corpo sem assinatura
    else:
        if linhas and not linhas[0].startswith("    "):
            completion_limpa = "\n".join(["    " + l for l in linhas])
        else:
            completion_limpa = completion

    return completion_limpa

# --- CARREGAMENTO ---
print(f"Carregando {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

problems = read_problems()
samples = []

print("Gerando amostras (greedy, temp=0.0)...")
for task_id in tqdm(problems):
    problem = problems[task_id]

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"Implement:\n{problem['prompt']}"},
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_input, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.0,   # greedy — padrão da literatura para pass@1
            do_sample=False,
        )

    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in raw_output:
        raw_completion = raw_output.split("assistant")[-1]
    else:
        raw_completion = raw_output

    final_completion = tratar_codigo(problem['prompt'], raw_completion)
    samples.append(dict(task_id=task_id, completion=final_completion))

write_jsonl(OUTPUT_FILE, samples)
print(f"Arquivo {OUTPUT_FILE} gerado com {len(samples)} amostras.")

In [ ]:
!evaluate_functional_correctness samples_base.jsonl

resultados base humaneval;

1.   teste 1: 0.6402439024390244;
2.   teste 2: 0.6402439024390244;


train rlif

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import re
import importlib

# --- 1. VACINAS DE ESTABILIDADE (CRÍTICO PARA RTX 3060 + WINDOWS) ---
torch.cuda.is_bf16_supported = lambda: False
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"

from transformers import modeling_utils
# Reload modeling_utils to discard the broken patch from the previous cell
importlib.reload(modeling_utils)

_original_to = modeling_utils.PreTrainedModel.to
def _patched_to(self, *args, **kwargs):
    # Remove the patch temporarily to prevent recursion
    modeling_utils.PreTrainedModel.to = _original_to
    try:
        return _original_to(self, *args, **kwargs)
    except ValueError as e:
        raise e
    finally:
        # Re-apply the patch
        modeling_utils.PreTrainedModel.to = _patched_to
modeling_utils.PreTrainedModel.to = _patched_to
# ---------------------------------------------------------------

from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- CONFIGURAÇÕES ---
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUTPUT_DIR = "qwen-rlif-v2"

# --- 2. FORMATAÇÃO DE DADOS ---
def format_data_func(example):
    gold_code = example.get('code', '')
    match = re.search(r"def\s+\w+\s*\(.*?\):", gold_code, re.DOTALL)
    signature = match.group(0) if match else "def solution():"

    prompt_raw = example.get('prompt', example.get('text', ''))

    system_msg = (
        "You are a Python coding engine. "
        "Respond with the COMPLETE Python function, including the def line and the indented body. "
        "Do NOT use input() or any form of user input. "
        "All values must come from function parameters. "
        "Do NOT include any explanation or markdown, only the raw Python function."
    )

    prompt_text = (
        f"<|im_start|>system\n{system_msg}<|im_end|>\n"
        f"<|im_start|>user\nImplement:\n{signature}\n\"\"\"\n{prompt_raw}\n\"\"\"\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    return {
        "prompt": prompt_text,
        "completion": gold_code,
    }

# --- 3. LIMPEZA DE MARKDOWN ---
# [FIX] O modelo estava gerando ```python ... ``` envolvendo o código.
# A confiança deve ser calculada sobre o código puro, não sobre os backticks.
def strip_markdown(text):
    if "```python" in text:
        text = text.split("```python")[1].split("```")[0]
    elif "```" in text:
        text = text.split("```")[1].split("```")[0]
    return text.strip()

# --- 4. EXECUÇÃO PRINCIPAL ---
def main():
    print(f"--- INICIANDO TREINO {OUTPUT_DIR} (RLIF v3 | ref_model CPU | markdown fix) ---")

    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    peft_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
        task_type=TaskType.CAUSAL_LM,
        lora_dropout=0.05,
    )

    training_args = GRPOConfig(
        output_dir=OUTPUT_DIR,
        learning_rate=5e-5,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_generations=4,
        max_prompt_length=600,
        max_completion_length=512,
        logging_steps=5,
        save_steps=50,
        fp16=True,
        beta=0.0005,
        use_vllm=False,
        report_to="none",
        temperature=0.7,
        seed=42,
        gradient_checkpointing=True,
        dataloader_pin_memory=False,
    )

    print("Carregando Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Carregando Modelo em Treinamento (GPU)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.pad_token_id = tokenizer.pad_token_id

    # [FIX] ref_model carregado explicitamente na CPU.
    # Antes o accelerate estava fazendo offload automático e parcial (meta device),
    # causando transferências CPU↔GPU lentas a cada step (~80s/step).
    # Na CPU o ref_model fica fixo e a transferência é feita uma vez por batch,
    # reduzindo o overhead significativamente.
    print("Carregando Modelo de Referência Congelado (CPU)...")
    ref_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # float32 na CPU — float16 não é suportado em todas CPUs
        device_map="cpu",
        trust_remote_code=True,
    )
    ref_model.config.pad_token_id = tokenizer.pad_token_id
    ref_model.eval()
    for param in ref_model.parameters():
        param.requires_grad_(False)
    print("ref_model congelado na CPU — sem competição de VRAM com o modelo em treino.")

    # --- 5. FUNÇÃO DE RECOMPENSA (RLIF) ---
    def reward_rlif_certainty(completions, prompts, **kwargs):
        rewards = []

        # [FIX] Remove markdown antes de calcular a confiança.
        # Sem isso, a log-prob dos backticks ```python``` dilui o sinal do código real.
        clean_completions = [strip_markdown(c) for c in completions]
        full_texts = [p + c for p, c in zip(prompts, clean_completions)]

        # Tokeniza na CPU — ref_model está na CPU
        inputs = tokenizer(
            full_texts, return_tensors="pt", padding=True, padding_side="right"
        )  # sem .to(device) — fica na CPU

        with torch.no_grad():
            outputs = ref_model(**inputs)
            logits = outputs.logits

        for i in range(len(full_texts)):
            prompt_inputs = tokenizer(prompts[i], add_special_tokens=False, return_tensors="pt")
            prompt_len = prompt_inputs.input_ids.shape[1]

            completion_logits = logits[i, prompt_len-1 : -1, :]
            input_ids = inputs.input_ids[i, prompt_len:]
            attention_mask = inputs.attention_mask[i, prompt_len:]

            valid_length = attention_mask.sum().item()
            if valid_length == 0:
                rewards.append(0.0)
                continue

            valid_logits = completion_logits[:valid_length]
            valid_ids = input_ids[:valid_length]

            token_log_probs = -F.cross_entropy(valid_logits, valid_ids, reduction='none')
            certainty_score = token_log_probs.mean().item()
            rewards.append(certainty_score)

            if i == 0:
                print(f"\nMONITOR RLIF | Confiança: {certainty_score:.4f}")
                print("-" * 50)
                print(clean_completions[i][:300])
                print("-" * 50 + "\n")

        return rewards

    print("Carregando Dataset...")
    dataset = load_dataset("mbpp", "sanitized", split="train")
    dataset = dataset.map(format_data_func, batched=False)

    print("Iniciando Trainer...")
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_rlif_certainty,
        args=training_args,
        train_dataset=dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    print("TREINO RLIF INICIADO!")
    trainer.train()

    print("Salvando modelo final...")
    trainer.save_model(OUTPUT_DIR)
    print(f"Modelo RLIF salvo em: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

--- INICIANDO TREINO qwen-rlif-v2 (RLIF v3 | ref_model CPU | markdown fix) ---
Carregando Tokenizer...


<string>:196: FutureWarning: The `max_prompt_length` argument is deprecated and will be removed in version 0.28.0. You should instead filter your dataset before training to ensure that prompts do not exceed your desired length.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


TREINO RLIF INICIADO!

MONITOR RLIF | Confiança: -0.1994
--------------------------------------------------
def remove_nested(test_tup):
    """
    Write a function to remove tuples from the given tuple.
    """
    result = tuple(item for item in test_tup if not isinstance(item, tuple))
    return result
--------------------------------------------------



Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
5,-0.006300
10,-0.091500
15,0.091800
20,0.064100
25,0.012600
30,0.011500
35,0.000600
40,-0.036600
45,-0.075400
50,0.010000



MONITOR RLIF | Confiança: -0.1611
--------------------------------------------------
def cummulative_sum(test_list):
    """
    Write a function to find the cumulative sum of all the values that are present in the given tuple list.
    """
    cumulative_sum = 0
    for value in test_list:
        cumulative_sum += value
    return cumulative_sum
--------------------------------------------------


MONITOR RLIF | Confiança: -0.1550
--------------------------------------------------
def noprofit_noloss(actual_cost, sale_amount):
    """
    Write a function to check whether the given amount has no profit and no loss
    """
    if actual_cost == sale_amount:
        return True
    else:
        return False
--------------------------------------------------


MONITOR RLIF | Confiança: -0.0961
--------------------------------------------------
def count_same_pair(nums1, nums2):
    """
    The input is defined as two lists of the same length. Write a function to count indices where th

avaliando rlif_code humaneval

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from human_eval.data import write_jsonl, read_problems
from tqdm import tqdm
import re

# --- CONFIGURAÇÕES ---
BASE_MODEL   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_PATH = "qwen-rlif-v2"   # pasta salva pelo treino RLIF
OUTPUT_FILE  = "samples_rlif.jsonl"
DEVICE = "cuda"

# --- SYSTEM PROMPT UNIFICADO ---
# Idêntico nos três scripts para garantir comparação justa.
SYSTEM_MSG = (
    "You are a Python coding engine. "
    "Respond with the COMPLETE Python function, including the def line and the indented body. "
    "Do NOT use input() or any form of user input. "
    "All values must come from function parameters. "
    "Do NOT include any explanation or markdown, only the raw Python function."
)

# --- FUNÇÃO DE PARSING UNIFICADA ---
# Idêntica nos três scripts.
def tratar_codigo(prompt_original, completion):
    # 1. Remove Markdown
    if "```python" in completion:
        completion = completion.split("```python")[1].split("```")[0]
    elif "```" in completion:
        completion = completion.split("```")[1].split("```")[0]

    completion = completion.strip()

    # 2. Descobre o nome da função pelo prompt do HumanEval
    match_def = re.search(r"def\s+(\w+)\s*\(", prompt_original)
    nome_funcao = match_def.group(1) if match_def else None

    linhas = completion.split('\n')

    # Caso A: modelo gerou a assinatura completa ("def func():")
    # Extrai só o corpo e reindenta com 4 espaços.
    if nome_funcao and any(nome_funcao in l and "def " in l for l in linhas):
        corpo = []
        capturando = False
        indentacao_base = None

        for linha in linhas:
            if f"def {nome_funcao}" in linha:
                capturando = True
                continue

            if capturando:
                if indentacao_base is None and linha.strip():
                    indentacao_base = len(linha) - len(linha.lstrip())

                if indentacao_base is not None:
                    if len(linha) - len(linha.lstrip()) >= indentacao_base or linha.strip() == "":
                        corpo.append(linha[indentacao_base:])
                    else:
                        break

        completion_limpa = "\n".join(["    " + l for l in corpo])

    # Caso B: modelo gerou só o corpo sem assinatura
    else:
        if linhas and not linhas[0].startswith("    "):
            completion_limpa = "\n".join(["    " + l for l in linhas])
        else:
            completion_limpa = completion

    return completion_limpa

# --- CARREGAMENTO ---
print(f"Carregando tokenizer: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print(f"Carregando modelo base: {BASE_MODEL}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
# resize_token_embeddings removido — nenhum token novo foi adicionado no treino

print(f"Acoplando adaptador RLIF: {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

problems = read_problems()
samples = []

print("Gerando amostras (greedy, temp=0.0)...")
for task_id in tqdm(problems):
    problem = problems[task_id]

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"Implement:\n{problem['prompt']}"},
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_input, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.0,   # greedy — padrão da literatura para pass@1
            do_sample=False,
        )

    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in raw_output:
        raw_completion = raw_output.split("assistant")[-1]
    else:
        raw_completion = raw_output

    final_completion = tratar_codigo(problem['prompt'], raw_completion)
    samples.append(dict(task_id=task_id, completion=final_completion))

write_jsonl(OUTPUT_FILE, samples)
print(f"Arquivo {OUTPUT_FILE} gerado com {len(samples)} amostras.")

Carregando tokenizer: Qwen/Qwen2.5-Coder-1.5B-Instruct...
Carregando modelo base: Qwen/Qwen2.5-Coder-1.5B-Instruct...
Acoplando adaptador RLIF: qwen-rlif-v2...
Gerando amostras (greedy, temp=0.0)...


100%|██████████| 164/164 [42:46<00:00, 15.65s/it]

Arquivo samples_rlif.jsonl gerado com 164 amostras.


In [ ]:
!evaluate_functional_correctness samples_rlif.jsonl

Reading samples...
164it [00:00, 9019.18it/s]
Running test suites...
100% 164/164 [00:01<00:00, 110.25it/s]
Writing results to samples_rlif.jsonl_results.jsonl...
100% 164/164 [00:00<00:00, 35327.71it/s]
{'pass@1': np.float64(0.6646341463414634)}


resultados rlif_code humaneval;

1.   teste 1: 0.6707317073170732;
2.   teste 2: 0.6646341463414634;





resultados code fine-tuning

In [ ]:
import pandas as pd

data = {
    "Modelo": ["Base", "RLVR", "RLIF"],
    "Teste 1": ["64.02%", "70.73%", "67.07%"],
    "Teste 2": ["64.02%", "67.07%", "66.46%"],
    "Média": ["64.02%", "68.90%", "66.77%"]
}

df = pd.DataFrame(data)
df

,Modelo,Teste 1,Teste 2,Média
0,Base,64.02%,64.02%,64.02%
1,RLVR,70.73%,67.07%,68.90%
2,RLIF,67.07%,66.46%,66.77%
